# pandas, Web Scraping und REST APIs

## DataPy - WiSe26/27 - Woche 5

### Data Science Programmierung mit Python

<style>
div[class*="OutputArea-output"] pre, .output_area pre { font-size: 15px !important; line-height: 1.25 !important; }
.output_html table, .dataframe { font-size: 14px !important; }
.text_cell_render table, .jp-RenderedMarkdown table { font-size: 20px !important; }
.output_html td, .output_html th, .dataframe td, .dataframe th { padding: 2px 8px !important; }
</style>

# Rückblick und Heute

- ✓ Einführung & Setup (Woche 1)
- ✓ Grundlagen Python I und II (Woche 2 und 3)
- ✓ NumPy: Arrays, Vektorisierung, Aggregation (Woche 4)

## Heute

1. pandas: Series und DataFrames
2. Joins, Concatenation und Group-by
3. Tabellen aus Webseiten lesen, Web Scraping mit Beautiful Soup
4. Rechtliche Aspekte des Scrapings
5. BONUS: Daten aus REST APIs, eine eigene API mit Flask

# pandas

- Etymologie: **pan**el **da**ta
- Entwickelt von **Wes McKinney**
- **DataFrames für Python**
- Lesen und Schreiben vieler Formate
- Sehr effiziente Operationen (viel effizienter als reines Python)
- Datenbankähnliche API
- Sehr beliebt bei Data Scientists

# Installation

Für diese Woche brauchen wir einige Pakete:

```
pip install pandas lxml html5lib beautifulsoup4 requests flask
```

**Hinweis:** Die Beispiele laufen mit **pandas 3**. Gegenüber pandas 2 ändert sich u. a.:

- Texte haben standardmäßig den Datentyp `str` (statt `object`)
- Ein Integer in `[]` bei einem Index aus Texten wird nicht mehr als Position gedeutet (siehe `loc` und `iloc`)

In [5]:
!pip install pandas lxml html5lib beautifulsoup4 requests flask

# pandas: Datenstrukturen

- **Series**: eindimensionales Array mit Index (denke: **Spalte** einer Tabelle/Datenbank)
- **DataFrame**: Tabelle (denke: **Datenbank**), eine oder mehrere Series nebeneinander

<img src="images/series_dataframe.png" alt="Eine Series mit Index und Werten und ein DataFrame aus mehreren Series" style="display:block; margin:0 auto; width:85%;">

# Die pandas Series

Eine Series ist ein **eindimensionales Array mit Index**.

In [6]:
import numpy as np
import pandas as pd

# eine pandas Series erzeugen
series_a = pd.Series([0.25, 0.5, 0.75, 1.0])
series_a

0    0.25
1    0.50
2    0.75
3    1.00
dtype: float64

In [7]:
series_a.index

RangeIndex(start=0, stop=4, step=1)

In [8]:
series_a.values

array([0.25, 0.5 , 0.75, 1.  ])

# Series erzeugen

Series lassen sich aus den meisten **Python-Collections** erzeugen, z. B. aus Listen, mit eigenem Index oder mit demselben Wert für alle Einträge.

In [9]:
# eine Liste mit Indizes
list_a = ['one', 'two', 'three']
list_b = ['index_one', 'index_two', 'index_three']

series_c = pd.Series(data=list_a, index=list_b)
series_c

index_one        one
index_two        two
index_three    three
dtype: str

In [10]:
# Series mit demselben Wert
pd.Series(5, index=[100, 200, 300])

100    5
200    5
300    5
dtype: int64

# Rechnen über den Index

Beim Rechnen werden Werte **über den Index** zugeordnet, nicht über die Position.

In [11]:
a = pd.Series({'apples': 1, 'lemons': 2, 'oranges': 0})
b = pd.Series({'lemons': 5, 'apples': 1, 'bananas': 3})

a + b

apples     2.0
bananas    NaN
lemons     7.0
oranges    NaN
dtype: float64

`bananas` und `oranges` kommen nur in einer Series vor, daher ist das Ergebnis `NaN` (*not a number*, fehlender Wert).

# Beispiel: Einwohner

Ein Dictionary wird zur Series, die **Schlüssel** werden zum **Index**.

In [64]:
berlin_population_dict = {
    '1917': 1681916,
    '1925': 4024286,
    '1933': 4242501,
    '1939': 4338756,
    '1945': 2807405,
    '1950': 3336026,
    '1970': 3208719,
    '1990': 3433695,
    '2000': 3382169,
    '2010': 3460725,
}

population = pd.Series(berlin_population_dict)

# Series indizieren

Series und Index können einen **Namen** bekommen. Achtung: Die Indizes sind hier Strings.

In [13]:
population.name = 'Population'
population.index.name = 'Year'

population.head(3)

Year
1917    1681916
1925    4024286
1933    4242501
Name: Population, dtype: int64

In [14]:
population['1917']

np.int64(1681916)

# Slicing und Filtern

In [15]:
population['1925':'1950']

Year
1925    4024286
1933    4242501
1939    4338756
1945    2807405
1950    3336026
Name: Population, dtype: int64

Beim Slicing mit **Labels** ist der Endpunkt **eingeschlossen** (`'1950'` ist im Ergebnis), anders als bei Positionen und Python-Listen.

In [16]:
# Boolean Indexing: nur Einträge, für die die Bedingung gilt
population[population > 3.4e6]

Year
1925    4024286
1933    4242501
1939    4338756
1990    3433695
2010    3460725
Name: Population, dtype: int64

In [17]:
# Bedingungen kombinieren: & (und), | (oder), je in Klammern
population[(population > 3.4e6) & (population < 4.1e6)]

Year
1925    4024286
1990    3433695
2010    3460725
Name: Population, dtype: int64

# Explizit vs. positionell

- **explizite Indizes**: das, was man als Index definiert hat
- **positionelle Indizes**: die Zeilennummer

Je nach Zugriff gilt das eine oder das andere. Um Verwechslungen zu vermeiden, gibt man den Typ mit `loc` (explizit) oder `iloc` (positionell) an.

In [18]:
data = pd.Series(['a', 'b', 'c'], index=[1, 3, 5])

# expliziter Index beim Indizieren
data[1]

'a'

In [19]:
# positioneller Index beim Slicing
data[1:3]

3    b
5    c
dtype: str

Mit einem **Index aus Texten** (`index=['1', '3', '5']`) wäre `data[1]` unklar. In pandas 2 galt `1` als Position (mit `FutureWarning`), **in pandas 3** gibt es einen `KeyError`. Darum: **immer `loc` oder `iloc` verwenden.**

# `loc` und `iloc`

In [20]:
data = pd.Series(['a', 'b', 'c'], index=[1, 3, 5])
# Werte:              a    b    c
# Index-Labels:       1    3    5

In [21]:
data.loc[1] # loc sucht nach dem Index-Label 1

'a'

In [22]:
data.loc[1:3]
# loc verwendet Index-Labels: von 1 bis 3.
# Der Endpunkt 3 ist bei loc eingeschlossen.

1    a
3    b
dtype: str

In [23]:
data.iloc[1:3]
# iloc verwendet Positionen, unabhängig von den Labels:
# Position 0 = a, Position 1 = b, Position 2 = c
# Es nimmt Position 1 bis vor Position 3, also b und c.

3    b
5    c
dtype: str

# Operationen auf Series

- Effiziente **arithmetische** und **Aggregationsoperationen**; **Kompatibel mit NumPy**

In [24]:
population_in_millions = population / 1e6
population_in_millions.head(3)

Year
1917    1.681916
1925    4.024286
1933    4.242501
Name: Population, dtype: float64

In [25]:
# Sortieren (hier: die drei kleinsten Werte)
population.sort_values().head(3)

Year
1917    1681916
1945    2807405
1970    3208719
Name: Population, dtype: int64

# Funktionen: `apply`

In [26]:
def some_function(v):
    '''
    Teilt durch eine Million und gibt den ganzzahligen Wert zurück
    '''
    return int(v / 1e6)


some_function(35 * 1e6)

35

In [27]:
population.apply(some_function).head(4) #bei einer Series wird some_function auf jeden einzelnen Wert angewendet:

Year
1917    1
1925    4
1933    4
1939    4
Name: Population, dtype: int64

# `apply` vs. Vektorisierung

In [28]:
s = pd.Series(range(int(1e6)))
%timeit s.apply(some_function)

172 ms ± 8.27 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [29]:
%timeit (s / 1e6).astype(int)

5.49 ms ± 437 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


Wann immer möglich: **vektorisierte Operationen** statt `apply` mit eigener Funktion.

# Fehlende Werte

Es gibt drei Hauptoptionen, mit fehlenden Werten umzugehen:

- Zeilen mit fehlenden Werten **löschen**
- fehlende Werte durch einen **Platzhalter** ersetzen
- fehlende Werte mit einem **ML-Modell imputieren** (geht über diese Einheit hinaus)

# Fehlende Werte: Beispiel

In [30]:
berlin_population_dict = {
    '1945': 2807405,
    '1950': 3336026,
    '1955': None,
    '1960': 3274016,
    '1965': np.nan,
    '1970': 3208719,
}

population_w_nans = pd.Series(berlin_population_dict)
population_w_nans

1945    2807405.0
1950    3336026.0
1955          NaN
1960    3274016.0
1965          NaN
1970    3208719.0
dtype: float64

# Fehlende Werte: löschen

In [31]:
population_w_nans.isnull()

1945    False
1950    False
1955     True
1960    False
1965     True
1970    False
dtype: bool

In [32]:
population_w_nans.dropna()

1945    2807405.0
1950    3336026.0
1960    3274016.0
1970    3208719.0
dtype: float64

# Fehlende Werte: ersetzen

In [33]:
# durch einen Platzhalter ersetzen
population_w_nans.fillna(0)

1945    2807405.0
1950    3336026.0
1955          0.0
1960    3274016.0
1965          0.0
1970    3208719.0
dtype: float64

In [34]:
# durch den Mittelwert ersetzen
population_w_nans.fillna(population_w_nans.mean())

1945    2807405.0
1950    3336026.0
1955    3156541.5
1960    3274016.0
1965    3156541.5
1970    3208719.0
dtype: float64

# Der pandas DataFrame

Ein DataFrame ist eine **tabellarische** Datenstruktur (oder Datenbank) in pandas, also eine oder mehrere aneinandergereihte Series.

In [35]:
some_numpy_array = np.arange(9).reshape((3, 3))
some_numpy_array

array([[0, 1, 2],
       [3, 4, 5],
       [6, 7, 8]])

In [36]:
df = pd.DataFrame(some_numpy_array,
                  index=['a', 'c', 'd'], #Name der Zeilen
                  columns=['Ohio', 'Texas', 'California']) #Name der Spalten
df

,Ohio,Texas,California
a,0,1,2
c,3,4,5
d,6,7,8


# DataFrame aus CSV

Mit `read_csv` lesen wir eine CSV-Datei ein. Damit das Beispiel ohne Zusatzdatei läuft, schreiben wir die Datei zuerst selbst:

In [37]:
import os

os.makedirs("data", exist_ok=True)
csv_path = os.path.join("data", "berlin_population.csv")
ps = population_w_nans.rename("population").rename_axis("year")
ps.to_csv(csv_path)

In [38]:
df_from_csv = pd.read_csv(csv_path)
df_from_csv

,year,population
0,1945,2807405.0
1,1950,3336026.0
2,1955,NaN
3,1960,3274016.0
4,1965,NaN
5,1970,3208719.0


# DataFrame IO

| Format | Beschreibung | Reader | Writer |
|--------|--------------|--------|--------|
| text | CSV | `read_csv` | `to_csv` |
| text | JSON | `read_json` | `to_json` |
| text | HTML | `read_html` | `to_html` |
| text | XML | `read_xml` | `to_xml` |
| binär | MS Excel | `read_excel` | `to_excel` |
| binär | Parquet | `read_parquet` | `to_parquet` |
| binär | Python Pickle | `read_pickle` | `to_pickle` |
| SQL | SQL | `read_sql` | `to_sql` |


# Beispieldaten: Fläche

In [39]:
area_km2 = {
    'Charlottenburg-Wilmersdorf': 64.72,
    'Friedrichshain-Kreuzberg': 20.16,
    'Lichtenberg': 52.29,
    'Marzahn-Hellersdorf': 61.74,
    'Mitte': 39.47,
    'Neukölln': 44.93,
    'Pankow': 103.01,
    'Reinickendorf': 89.46,
    'Spandau': 91.91,
    'Steglitz-Zehlendorf': 102.5,
    'Tempelhof-Schöneberg': 53.09,
    'Treptow-Köpenick': 168.42}

# Beispieldaten: Einwohner

In [40]:
population_2010 = {
    'Charlottenburg-Wilmersdorf': 320014,
    'Friedrichshain-Kreuzberg': 268831,
    'Lichtenberg': 259881,
    'Marzahn-Hellersdorf': 248264,
    'Mitte': 332100,
    'Neukölln': 310283,
    'Pankow': 368956,
    'Reinickendorf': 240454,
    'Spandau': 225420,
    'Steglitz-Zehlendorf': 293989,
    'Tempelhof-Schöneberg': 335060,
    'Treptow-Köpenick': 241335}

berlin_population_by_borough = {
    'Area in km²': area_km2,
    'Population 30 September 2010': population_2010}

# DataFrame aus Dicts

Fläche und Einwohner der **12 Bezirke** (30. September 2010). Wie Series haben DataFrames **Indizes** (Zeilen und Spalten).

In [41]:
df = pd.DataFrame(berlin_population_by_borough)
df

,Area in km²,Population 30 September 2010
Charlottenburg-Wilmersdorf,64.72,320014
Friedrichshain-Kreuzberg,20.16,268831
Lichtenberg,52.29,259881
Marzahn-Hellersdorf,61.74,248264
Mitte,39.47,332100
Neukölln,44.93,310283
Pankow,103.01,368956
Reinickendorf,89.46,240454
Spandau,91.91,225420
Steglitz-Zehlendorf,102.50,293989


In [42]:
# Zugriff über Zeilen- und Spaltenindex
df.loc['Friedrichshain-Kreuzberg', 'Area in km²']

np.float64(20.16)

In [43]:
# eine ganze Spalte mit loc (hier nur die ersten drei Zeilen)
df.loc[:, 'Area in km²'].head(3)

Charlottenburg-Wilmersdorf    64.72
Friedrichshain-Kreuzberg      20.16
Lichtenberg                   52.29
Name: Area in km², dtype: float64

In [44]:
# eine ganze Spalte (= Series)
df['Area in km²'].head(3)

Charlottenburg-Wilmersdorf    64.72
Friedrichshain-Kreuzberg      20.16
Lichtenberg                   52.29
Name: Area in km², dtype: float64

# Boolean Indexing - Zeilenfiltern anhand einer Bedingung

In [45]:
col = 'Population 30 September 2010'
df[df[col] > 3e5] #prüft für jede Zeile, ob der Wert in dieser Spalte größer als 300 000
#Das Ergebnis ist eine Serie mit True oder False, etwa:
# a     True
# c    False
# d     True
#Dann verwendet pandas diese Wahr/Falsch-Werte als Filter
#Es bleiben nur die Zeilen mit True übrig. Also: alle Einträge, deren Bevölkerung am 30. September 2010 über 300.000 lag.

,Area in km²,Population 30 September 2010
Charlottenburg-Wilmersdorf,64.72,320014
Mitte,39.47,332100
Neukölln,44.93,310283
Pankow,103.01,368956
Tempelhof-Schöneberg,53.09,335060


In [46]:
# boolesche Zeilenauswahl mit Spaltenauswahl
df.loc[df[col] > 3e5, col] #filtert zuerst die Zeilen und wählt dann nur die angegebene Spalte aus.

#Das Ergebnis ist eine Series statt eines ganzen DataFrames:

Charlottenburg-Wilmersdorf    320014
Mitte                         332100
Neukölln                      310283
Pankow                        368956
Tempelhof-Schöneberg          335060
Name: Population 30 September 2010, dtype: int64

In [47]:
# positionell mit iloc: die ersten zwei Zeilen, die erste Spalte
df.iloc[:2, 0]

Charlottenburg-Wilmersdorf    64.72
Friedrichshain-Kreuzberg      20.16
Name: Area in km², dtype: float64

# Daten erkunden

In [48]:
df.head(3)

,Area in km²,Population 30 September 2010
Charlottenburg-Wilmersdorf,64.72,320014
Friedrichshain-Kreuzberg,20.16,268831
Lichtenberg,52.29,259881


In [49]:
df.describe()

,Area in km²,Population 30 September 2010
count,12.000000,12.000000
mean,74.308333,287048.916667
std,39.552859,46074.394769
min,20.160000,225420.000000
25%,50.450000,246531.750000
50%,63.230000,281410.000000
75%,94.557500,323035.500000
max,168.420000,368956.000000


# `info`: Spalten und Typen

In [50]:
df.info()

<class 'pandas.DataFrame'>
Index: 12 entries, Charlottenburg-Wilmersdorf to Treptow-Köpenick
Data columns (total 2 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Area in km²                   12 non-null     float64
 1   Population 30 September 2010  12 non-null     int64  
dtypes: float64(1), int64(1)
memory usage: 588.0+ bytes


# Operationen auf DataFrames

- Alle **Series-Operationen** funktionieren auf DataFrame-Spalten
- DataFrames unterstützen alle gängigen **Datenbankoperationen** und mehr
- Neue Spalte berechnen, sortieren, hier: die vier am dünnsten besiedelten Bezirke

In [51]:
df['Density'] = df['Population 30 September 2010'] / df['Area in km²']
df = df.sort_values(by=['Density'])
df.head(4)

,Area in km²,Population 30 September 2010,Density
Treptow-Köpenick,168.42,241335,1432.935518
Spandau,91.91,225420,2452.616690
Reinickendorf,89.46,240454,2687.838140
Steglitz-Zehlendorf,102.50,293989,2868.185366


# Joins mit pandas

Zwei Tabellen werden über eine gemeinsame Spalte (**Key**) verbunden.

In [52]:
df1 = pd.DataFrame({'key': ['b', 'b', 'a', 'c', 'a', 'a', 'b'],
                    'data1': range(7)})
df2 = pd.DataFrame({'key': ['a', 'b', 'd'], 'data2': range(3)})
df1

,key,data1
0,b,0
1,b,1
2,a,2
3,c,3
4,a,4
5,a,5
6,b,6


In [53]:
df2

,key,data2
0,a,0
1,b,1
2,d,2


# Inner Join und Left Join

**Inner**: nur Keys, die in **beiden** Tabellen vorkommen. **Left**: alle Keys der linken Tabelle, fehlende Werte werden mit `NaN` gefüllt.

In [54]:
pd.merge(df1, df2, on='key')

,key,data1,data2
0,b,0,1
1,b,1,1
2,a,2,0
3,a,4,0
4,a,5,0
5,b,6,1


In [55]:
pd.merge(df1, df2, on='key', how='left')

,key,data1,data2
0,b,0,1.0
1,b,1,1.0
2,a,2,0.0
3,c,3,NaN
4,a,4,0.0
5,a,5,0.0
6,b,6,1.0


# Outer Join

**Alle** Keys aus beiden Tabellen, fehlende Werte werden mit `NaN` gefüllt:

In [56]:
pd.merge(df1, df2, on='key', how='outer')

,key,data1,data2
0,a,2.0,0.0
1,a,4.0,0.0
2,a,5.0,0.0
3,b,0.0,1.0
4,b,1.0,1.0
5,b,6.0,1.0
6,c,3.0,NaN
7,d,NaN,2.0


# Join-Typen im Überblick

<img src="images/join_types.png" alt="Welche Keys die Join-Typen inner, left, right und outer im Ergebnis behalten" style="display:block; margin:0 auto; width:100%;">

# Concatenation

Zur Erinnerung aus Woche 4: `np.concatenate` hängt Arrays aneinander. In pandas macht das `pd.concat`:

In [57]:
s1 = pd.Series([0, 1], index=['a', 'b'])
s2 = pd.Series([2, 3, 4], index=['c', 'd', 'e'])
s3 = pd.Series([5, 6], index=['f', 'g'])
pd.concat([s1, s2, s3])

a    0
b    1
c    2
d    3
e    4
f    5
g    6
dtype: int64

# Group-by und Aggregationen

Auch bekannt als **split-apply-combine**:

<img src="images/split_apply_combine.png" alt="Split, Apply und Combine am Beispiel groupby und mean" style="display:block; margin:0 auto; width:100%;">

# Group-by: Beispieldaten

In [58]:
np.random.seed(0)   # für reproduzierbare Zufallszahlen

df = pd.DataFrame({'key1': ['a', 'a', 'b', 'b', 'a'],
                   'key2': ['one', 'two', 'one', 'two', 'one'],
                   'data1': np.random.randn(5),
                   'data2': np.random.randn(5)})
df

,key1,key2,data1,data2
0,a,one,1.764052,-0.977278
1,a,two,0.400157,0.950088
2,b,one,0.978738,-0.151357
3,b,two,2.240893,-0.103219
4,a,one,1.867558,0.410599


# Group-by: Aggregation

In [59]:
grouped = df['data1'].groupby(df['key1'])
grouped.mean()

key1
a    1.343923
b    1.609816
Name: data1, dtype: float64

In [60]:
# mehrere Keys und agg
df.groupby(['key1', 'key2'])['data1'].agg(['mean', 'sum']).rename(
    columns={'mean': 'my_mean', 'sum': 'my_sum'})

my_mean    my_sum
key1 key2                    
a    one   1.815805  3.631610
     two   0.400157  0.400157
b    one   0.978738  0.978738
     two   2.240893  2.240893

# PAUSE - dann Teil 2 -

# Daten aus dem Web

- Viele Daten liegen **nicht als Datei**, sondern auf Webseiten oder hinter Schnittstellen
- Zwei Wege: **Web Scraping** (HTML auslesen) oder **API** (strukturierte Daten abfragen)
- Immer zuerst prüfen: Gibt es eine **API** oder einen **Download**? Das ist stabiler als Scraping

In [61]:
import requests
from io import StringIO

# manche Server lehnen Anfragen ohne User-Agent ab
HEADERS = {"User-Agent": "Mozilla/5.0 (DataPy-Kurs, BHT Berlin)"}

# Tabellen: `read_html`

In [62]:
url = "https://en.wikipedia.org/wiki/Berlin" #Adresse der gewünschten Webseite

#request sendet eine HTTP-`GET`-Anfrage an diese Adresse
#headers=HEADERS:sendet zusätzliche Angaben mit, oft z. B. einen User-Agent, der beschreibt, welches Programm die Anfrage stellt
#timeout=10: Nach maximal 10 Sekunden wird abgebrochen, falls keine Antwort kommt.
response = requests.get(url, headers=HEADERS, timeout=10) 

#enthält die Antwort des Servers – etwa Inhalt, Header und Status.
response.status_code

200

# typische HTTP-Statuscodes

| Code | Bedeutung |
|---|---|
| 1xx | Informational |
| 2xx | Success (z. B. `200 OK`) |
| 3xx | Redirection |
| 4xx | Client Error (z. B. `404 Not Found`, `403 Forbidden`) |
| 5xx | Server Error (z. B. `500 Internal Server Error`) |

In [63]:
# match: nur Tabellen, die diesen Text enthalten


#response.text enthält den HTML-Quelltext der heruntergeladenen Wikipedia-Seite.
#StringIO(...) behandelt diesen Text wie eine Datei, damit pandas ihn einlesen kann.
#pd.read_html(...) sucht HTML-Tabellen und wandelt sie in pandas-DataFrames um.
#match="±%" berücksichtigt nur Tabellen, die den Text unter match enthalten. Das grenzt die passende Tabelle ein.
tables = pd.read_html(StringIO(response.text), match="Population", header=0) #liste der tabellen mit dem match

df_pop = tables[0] #tables[0] nimmt die erste gefundene Tabelle und speichert sie als df_pop.
df_pop.head() #die ersten 5 Zeilen des DataFrames

ImportError: `Import lxml` failed.  Use pip, conda, or your preferred package management tool to install the lxml package.

# Daten bereinigen
--> Webdaten müssen fast immer bereinigt werden.

```
df_pop = tables[0]  # wenn hier zB. table Year und Pop. stehen udn wir sie löschen wollen
print(df_pop.columns.tolist())

df_pop = df_pop[["Year", "Pop."]]
df_pop.tail(2)
```

# Beautiful Soup

`read_html` findet nur Tabellen. Für alles andere: **Beautiful Soup** (`bs4`)

--> eine Python-Bibliothek zum Auslesen und Durchsuchen von HTML (Webseiten) oder XML. Sie macht aus HTML-Text eine Struktur, in der du Tags wie < p >, < a > oder < title > gezielt finden kannst.

In [ ]:
from bs4 import BeautifulSoup

a_simple_webpage = """
<html>
  <head><title>Meine Seite</title></head>
  <body>
    <p>Erster Absatz.</p>
    <p id="second_paragraph">Zweiter Absatz mit
      <a href="https://www.bht-berlin.de">Link</a>.</p>
  </body>
</html>
"""

soup = BeautifulSoup(a_simple_webpage, "html.parser") #"html.parser" ist der eingebaute Parser, der das HTML interpretiert
[child.name for child in soup.children] #children liefert die direkten Kindelemente des Dokuments. Die List Comprehension nimmt von jedem Element dessen Tag-Namen.

#zum Ergebnis: None steht hier für Leerzeilen bzw. Text zwischen den Tags. Das eigentliche HTML-Element heißt html.

# Soup: Elemente finden

In [ ]:
soup.find("title").text

In [ ]:
for p in soup.find_all("p"): #alle paragraphs finden
    print(p.get_text().strip())

In [ ]:
for a in soup.find_all("a"): #link-tags
    print(a.get_text(), "->", a.get("href"))

In [ ]:
soup.find_all(id="second_paragraph")

# Echte Webseite

In [ ]:
url = "https://de.wikipedia.org/wiki/Berlin"
page = requests.get(url, headers=HEADERS, timeout=10)
print(page.status_code)

soup = BeautifulSoup(page.content, "html.parser")
print(soup.title.get_text())
print(soup.find("h1").get_text())

In [ ]:
# eine Seite, die es nicht gibt
url = "https://de.wikipedia.org/wiki/Diese_Seite_gibt_es_nicht_12345"
missing = requests.get(url, headers=HEADERS, timeout=10)
missing.status_code

# Crawling und Scrapy

- **Crawling**: Links automatisch folgen und viele Seiten abrufen
- Für große Projekte gibt es das Framework **Scrapy** ([scrapy.org](https://scrapy.org/))
- Beim Crawlen: **Wartezeiten** einbauen, `robots.txt` beachten, Server nicht überlasten

# Rechtliche Aspekte

*Hinweis: Dies ist keine Rechtsberatung.*

- Datenbanken können durch das **Datenbankherstellerrecht** geschützt sein (§§ 87a ff. UrhG): geschützt ist die Investition in die Datenbank, nicht der einzelne Inhalt
- Entnahme **wesentlicher Teile** ist ohne Erlaubnis nicht zulässig, auch wiederholte Entnahme **unwesentlicher Teile**, wenn sie auf einen wesentlichen Teil hinausläuft
- Was „wesentlich“ ist, entscheidet sich im **Einzelfall**; auch Anteile in der Größenordnung von **10 %** wurden von Gerichten schon so eingestuft
- **Nutzungsbedingungen** und `robots.txt` der Seite prüfen
- Für **wissenschaftliche Forschung** gibt es Ausnahmen für Text und Data Mining (§ 60d UrhG)
- Personenbezogene Daten: zusätzlich **DSGVO** beachten

# REST APIs

- **API** = Application Programming Interface: definierte Schnittstelle zwischen Programmen
- **REST**: Zugriff über HTTP auf Ressourcen mit URLs, Antwort meist als **JSON**
- Anfrage: `GET https://server/ressource?parameter=wert`
- Antwort: Statuscode + Daten

<img src="images/rest_request.png" alt="Ablauf einer REST-Anfrage: Client sendet GET-Request, Server antwortet mit JSON" style="display:block; margin:0 auto; width:100%;">

# Beispiel: AQICN

Die API [waqi.info](https://aqicn.org/api/) liefert den Air Quality Index (AQI) für Städte.

- Dafür braucht man einen **persönlichen Token** (kostenlos: [aqicn.org/data-platform/token](https://aqicn.org/data-platform/token/))
- Token **nie** in öffentlichen Code oder Folien schreiben

In [ ]:
# eigenen Token als Umgebungsvariable setzen oder hier eintragen
accesstoken = os.environ.get("WAQI_TOKEN", "DEIN_TOKEN")


def get_air_quality_data(city, accesstoken):
    url = f"https://api.waqi.info/feed/{city}/?token={accesstoken}"
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    result = response.json()
    if result["status"] != "ok":
        print("Fehler:", result["data"])
        return None
    return result["data"]

In [ ]:
aq_data = get_air_quality_data("berlin", accesstoken)
if aq_data:
    print("AQI:", aq_data["aqi"])
    print("Station:", aq_data["city"]["name"])
    print("Zeit:", aq_data["time"]["s"])

# Aufbau der JSON-Antwort

```json
{"status": "ok",
 "data": {"aqi": 42,
          "city": {"name": "Berlin ...", "geo": [52.5, 13.4]},
          "iaqi": {"pm25": {"v": 42}, "no2": {"v": 12.1}},
          "time": {"s": "2026-10-06 12:00:00"}}}
```

- `status`: `"ok"` oder `"error"`
- `data["aqi"]`: Gesamtindex, `data["iaqi"]`: einzelne Schadstoffe

# BONUS: Eigene API mit Flask

**Flask** ist ein leichtgewichtiges Python-Framework für Webanwendungen und APIs. Beispiel: Nachrichten-Überschriften als JSON anbieten (mit kleinen Beispieldaten).

1. Datei `newsapi.py` anlegen (nächste Folie)
2. Im **Terminal** starten: `python newsapi.py` (läuft, bis man es beendet)
3. Im Notebook abfragen

# `newsapi.py`

In [ ]:
%%writefile newsapi.py
from flask import Flask, jsonify

app = Flask(__name__)
news = {"zeit": ["Schlagzeile 1", "Schlagzeile 2"],
        "taz": ["Schlagzeile 1", "Schlagzeile 2"]}


@app.route("/newsapi/<newspaper_id>")
def get_news(newspaper_id):
    if newspaper_id not in news:
        return jsonify({"error": "unbekannte Zeitung"}), 404
    return jsonify({"headlines": news[newspaper_id]})


if __name__ == "__main__":
    app.run(host="127.0.0.1", port=5001)

# API abfragen

In [ ]:
r = requests.get("http://127.0.0.1:5001/newsapi/zeit", timeout=5)
print(r.status_code)
r.json()

In [ ]:
r = requests.get("http://127.0.0.1:5001/newsapi/unbekannt", timeout=5)
r.status_code

# Nützliche Ressourcen

- pandas: [pandas.pydata.org/docs](https://pandas.pydata.org/docs/)
- Beautiful Soup: [crummy.com/software/BeautifulSoup/bs4/doc](https://www.crummy.com/software/BeautifulSoup/bs4/doc/)
- requests: [requests.readthedocs.io](https://requests.readthedocs.io/)
- Flask: [flask.palletsprojects.com](https://flask.palletsprojects.com/)
- AQICN API: [aqicn.org/api](https://aqicn.org/api/)
- Buch: [Python Data Science Handbook (J. VanderPlas)](https://jakevdp.github.io/PythonDataScienceHandbook/)

## Nächste Woche

**Projekt-Kick-off:** Vorstellung des Semesterprojekts, Ideenfindung und Datensatzsuche